<h3>Verify qdk-chemistry installation</h3>

In [ ]:
# Verify qdk-chemistry installation
import qdk_chemistry
print(f"qdk-chemistry version: {qdk_chemistry.__version__}")

<h3>End-to-end workflow preview</h3>

In [ ]:
# End-to-end workflow preview
# Each block is labelled with the chapter that covers it in depth.
# Uses a minimal 2e/2o active space and HF trial state for speed.
# Chapter 3 and 5 cover proper active space selection and multi-reference state prep.

from pathlib import Path
import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit
from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

# ── Chapter 1: load molecule and run SCF ──────────────────────────────────────
structure = Structure.from_xyz_file(Path("../examples/data/stretched_n2.structure.xyz"))
E_hf, wfn_hf = create("scf_solver").run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)
print(f"[Ch.1] HF energy:            {E_hf:.6f} Hartree")

# ── Chapter 2: localize orbitals ──────────────────────────────────────────────
wfn_valence = create("active_space_selector", "qdk_valence",
                     num_active_electrons=2,
                     num_active_orbitals=2).run(wfn_hf)
indices = wfn_valence.get_orbitals().get_active_space_indices()
wfn_loc = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(wfn_valence, *indices)
print(f"[Ch.2] Orbitals localized")

# ── Chapter 3: active space — 2 electrons in 2 orbitals (HOMO/LUMO) ──────────
n_alpha, n_beta = wfn_loc.get_active_num_electrons()
active_orbitals = wfn_loc.get_orbitals()
print(f"[Ch.3] Active space:          {n_alpha + n_beta} electrons, 2 orbitals → 4 qubits (JW)")

# ── Chapter 4: build Hamiltonian and map to qubits ────────────────────────────
hamiltonian = create("hamiltonian_constructor").run(active_orbitals)
qubit_hamiltonian = create("qubit_mapper", "qiskit", encoding="jordan-wigner").run(hamiltonian)
print(f"[Ch.4] Qubit Hamiltonian:     {len(qubit_hamiltonian.pauli_strings)} Pauli strings")

# ── Chapter 5: prepare trial state from HF wavefunction ──────────────────────
state_prep_circuit = create("state_prep", "sparse_isometry_gf2x").run(wfn_loc)
print(f"[Ch.5] State prep circuit ready")

# ── Chapter 6: iterative QPE ──────────────────────────────────────────────────
evolution_builder = create("time_evolution_builder", "trotter")
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")
iqpe = create("phase_estimation", "iterative", num_bits=2, evolution_time=0.5, shots_per_bit=1)
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=create("circuit_executor", "qdk_full_state_simulator"),
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)
estimated_energy = result.raw_energy + hamiltonian.get_core_energy()
print(f"\n[Ch.6] HF reference energy:   {E_hf:.6f} Hartree")
print(f"[Ch.6] QPE estimated energy:  {estimated_energy:.6f} Hartree")
print(f"[Ch.6] Note: full precision and multi-reference state prep covered in Ch.5-6")


<h3>Explore the algorithm registry</h3>

In [ ]:
# Explore the algorithm registry
from qdk_chemistry.algorithms import available

# Snapshot the registry before loading any plugins
registry_before_plugins = {k: list(v) for k, v in available().items()}

print("Algorithm types available (no plugins loaded):")
for alg_type, implementations in registry_before_plugins.items():
    print(f"  {alg_type}: {implementations}")

<h3>Load plugins and confirm registry</h3>

In [ ]:
# Load plugins and confirm registry
import qdk_chemistry.plugins.pyscf   # enables: scf_solver, orbital_localizer, stability_checker
import qdk_chemistry.plugins.qiskit  # enables: qubit_mapper (qiskit encoding), state_prep (qiskit_regular_isometry)

registry_after_plugins = available()

# Confirm the registry names are unchanged: plugins affect execution, not registration
print("Registry is identical before and after plugin import:", registry_before_plugins == registry_after_plugins)
print()
print("Algorithm types and implementations (with plugins loaded):")
for alg_type, implementations in registry_after_plugins.items():
    print(f"  {alg_type}: {implementations}")

<h3>Query specific algorithm types</h3>

In [ ]:
# Query specific algorithm types
scf_implementations = available("scf_solver")
selector_implementations = available("active_space_selector")

print(f"scf_solver implementations:       {scf_implementations}")
print(f"active_space_selector implementations: {selector_implementations}")

<h3>Print settings table for SCF solver</h3>

In [ ]:
# Print settings table for SCF solver
from qdk_chemistry.algorithms import inspect_settings, print_settings

# Print a formatted settings table for the default SCF solver
print("Settings for scf_solver / qdk:")
print_settings("scf_solver", "qdk")

<h3>Inspect settings programmatically (qdk)</h3>

In [ ]:
# Inspect settings programmatically (qdk)
settings_info = inspect_settings("scf_solver", "qdk")

print(f"{'Name':<30} {'Type':<10} {'Default':<15} {'Limits'}")
print("-" * 75)
for name, python_type, default, description, limits in settings_info:
    print(f"{name:<30} {str(python_type):<10} {str(default):<15} {str(limits) if limits else ''}")

<h3>Inspect settings programmatically (pyscf)</h3>

In [ ]:
# Inspect settings programmatically (pyscf)
settings_info = inspect_settings("scf_solver", "pyscf")

print(f"{'Name':<25} {'Type':<10} {'Default':<15} {'Limits'}")
print("-" * 70)
for name, python_type, default, description, limits in settings_info:
    print(f"{name:<25} {str(python_type):<10} {str(default):<15} {str(limits) if limits else ''}")

<h3>Create algorithm instances with custom settings</h3>

In [ ]:
# Create algorithm instances with custom settings
from qdk_chemistry.algorithms import create

# Create the default SCF solver (no name = use the registry default)
scf_default = create("scf_solver")
print("Default SCF solver type:", type(scf_default).__name__)
print("Default settings:", scf_default.settings().to_dict())

print()

# Create a PySCF solver with custom settings passed at creation time
scf_tight = create("scf_solver", "pyscf", max_iterations=100, convergence_threshold=1e-8)
print("Custom SCF solver settings:", scf_tight.settings().to_dict())

<h3>Prerequisite: environment check</h3>

In [ ]:
# Prerequisite: environment check
from pathlib import Path
import qdk_chemistry
import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit
from qdk_chemistry.algorithms import available, create

# Check plugins unlocked the expected types
registry = available()
required_types = [
    "scf_solver", "orbital_localizer", "active_space_selector",
    "hamiltonian_constructor", "qubit_mapper", "state_prep",
    "phase_estimation", "circuit_executor",
]
for t in required_types:
    assert t in registry, f"Missing algorithm type: {t}"

# Check N2 data file is present (adjust path if running from a different directory)
n2_path = Path("../examples/data/stretched_n2.structure.xyz")
assert n2_path.exists(), f"N2 structure file not found at {n2_path.resolve()}"

print("All checks passed. Environment is ready — proceed to Chapter 1.")